# Comparing S(Q,$\omega$)

**OK so we are pretty confident our simulation is showing that the rotation of the liquid benzene is anisotropic! We now need to understand how reflective of the experiment is our simulation and where does it deviate?**


In order to analyse S(Q,$\omega$) we need to calculate it, hopefully you are now very aware that this can be done using MDANSE!

Some points to consider for the calculaton of S(Q,$\omega$) of this system:

* What kind of length scale I am interested in?
* What kind of energy exchange I am interested in?
* What kind of simulation sampling do I need to match this energy exchange / length scale and ensure my results are converged? 

The [documentation](https://mdanse.readthedocs.io/en/latest/how_to_guides/H_conv.html) can be very helpful for this!


Have a crack at running the 1ns trajectory through MDANSE to calculate $S_{inc}$(Q,$\omega$). 
(If this is taking ages the output has also been provided for you in ```Data/benzene```)


For reference we are going to be comparing this to two experimental QENS datasets both taken from the ISIS Neutron and Muon Source 

One from the unpolarised indirect geometry spectrometer [IRIS](https://www.isis.stfc.ac.uk/instruments/iris/) using the PG 002 crystal and one from the polarised direct geomtry spectrometer (3.7meV incident energy) [LET](https://www.isis.stfc.ac.uk/instruments/let/)  (Q ~ 0.5-2 Å$^{-1}$, $-2\leq\omega\leq 2$ meV).



Once you have had a go at generating S(Q,w), we can load and plot the output with the below code!


In [ ]:
import h5py
from pathlib import Path
import scipp as sc
import matplotlib.pyplot as plt
import numpy as np
from _fig_params import colors
import sys

DATA = Path("../../Data/benzene")
MDA = DATA / "incoh.mda"
COMP = "total"
HBAR = 0.6582119569  # meV per rad/ps

inc_dict = {}
with h5py.File(MDA, "r") as f:
    inc_dict["q"] = f["disf/axes/q"][...] / 10
    inc_dict["omega"] = f["disf/axes/omega"][...] * HBAR
    inc_dict["S"] = f[f"disf/s(q,f)/{COMP}"][...] / HBAR


def create_scipp_array(S, q, omega):
    half_bin_omega = (np.diff(omega) / 2)[0]
    omega = np.append(omega - half_bin_omega, omega[-1] + half_bin_omega)

    half_bin_q = (np.diff(q) / 2)[0]
    q = np.append(q - half_bin_q, q[-1] + half_bin_q)

    sqw = sc.DataArray(
        data=sc.array(dims=["q", "omega"], values=S),
        coords={
            "q": sc.array(dims=["q"], values=q, unit="1/Angstrom"),
            "omega": sc.array(dims=["omega"], values=omega, unit="meV"),
        },
    )
    return sqw


sqw = create_scipp_array(inc_dict["S"], inc_dict["q"], inc_dict["omega"])
q_bins = sc.linspace("q", 0.6, 1.8, num=50, unit="1/Angstrom")
omega_bins = sc.linspace("omega", -2.25, 2.25, num=1000, unit="meV")
bin_dict = {"q": q_bins, "omega": omega_bins}
sqw.rebin(**bin_dict).plot()

We can also have a look at a few individual Q, we see the spectra widening and the intensity dropping as a function of Q

In [ ]:
q_bins = sc.linspace("q", 0.6, 1.8, num=20, unit="1/Angstrom")
omega_bins = sc.linspace("omega", -2.25, 2.25, num=1000, unit="meV")
bin_dict = {"q": q_bins, "omega": omega_bins}
inc = sqw.rebin(**bin_dict)

fig, ax = plt.subplots(figsize=(4, 4))
ax = [ax]

for i in [0, 2, 5, 10, 15]:
    ax[0].plot(
        sc.midpoints(inc.coords["omega"]), inc.data["q", i], color=colors[1], alpha=0.5
    )

ax[0].set_xlabel(r"$\omega$ / meV")

Have a play binning some of the data and making some plots

The next question is then, how does our data compare to a real QENS experiment?

**Let's load in our IRIS QENS data.**

In [ ]:
IRIS_FOLDER = DATA / "IRIS/"
sys.path.append(str(IRIS_FOLDER))
from iris import load_iris


benzene = load_iris(IRIS_FOLDER / "IRIS00107089.nxs")
bkg = load_iris(IRIS_FOLDER / "IRIS00107097.nxs")
reso = load_iris(IRIS_FOLDER / "IRIS00107073.nxs")

fig, ax = plt.subplots(2, 3, figsize=(8, 6))
ax = ax.ravel()
benzene.plot(ax=ax[0])
reso.plot(ax=ax[1])
bkg.plot(ax=ax[2])

benzene.nansum("q").plot(ax=ax[3])
reso.nansum("q").plot(ax=ax[4])
bkg.nansum("q").plot(ax=ax[5])

plt.show()

Do a background intensity subtraction for both the benzene and the vanadium:

In [ ]:
benzene.data -= bkg.data
reso.data -= bkg.data

Now lets compare our simulated spectra to our experimental spectra across Q! (I hate 2D spectra for this so would normally use lots of 1D)

In [ ]:
fig, ax = plt.subplots(3, 4, figsize=(10, 7.5))
ax = ax.ravel()

q_bins = sc.linspace("q", 0.5, 1.8, num=13, unit="1/Angstrom")
omega_bins = sc.linspace("omega", -0.5, 0.5, num=1000, unit="meV")
bin_dict = {"q": q_bins, "omega": omega_bins}

sqw = sqw.rebin(**bin_dict)
benzene = benzene.rebin(**bin_dict)

for i in range(len(q_bins) - 1):
    ax[i].plot(
        sc.midpoints(sqw.coords["omega"]), sqw.data["q", i] * 0.6, color=colors[0]
    )
    ax[i].errorbar(
        sc.midpoints(benzene.coords["omega"]).values,
        benzene.data["q", i].values,
        benzene.variances[i] ** 0.5,
        color=colors[1],
        alpha=0.4,
    )
    ax[i].set_title(f"q_mid: {q_bins.values[i]:.2f}")
    ax[i].set_xlim(-0.4, 0.4)

Hmmm our benzene looks uniformly too narrow, this is very obvious at low Q, why do you think that is?



(We need to include a resolution function!)

In [ ]:
from scipy.signal import fftconvolve


def convolve(sqw, reso):
    r = np.nan_to_num(reso.values)
    k = r / r.sum(axis=1, keepdims=True)
    out = sqw.copy()
    out.values = fftconvolve(
        np.nan_to_num(sqw.values), np.nan_to_num(k), mode="same", axes=1
    )
    return out


def conv_and_norm(sqw, reso, exp, bin_dict, pad=200):
    e = bin_dict["omega"]
    exp = exp.rebin(**bin_dict)
    reso = reso.rebin(**bin_dict)
    sqw = sqw.rebin(**bin_dict)

    sqw_c = convolve(sqw.rebin(q=bin_dict["q"], omega=e), reso)
    sqw_c = sqw_c / sqw_c.max() * np.nan_to_num(exp.values).max()
    return sqw_c


sqw_c = conv_and_norm(sqw, reso, benzene, bin_dict)

In [ ]:
fig, ax = plt.subplots(3, 4, figsize=(10, 7.5))
ax = ax.ravel()

for i in range(len(q_bins) - 1):
    ax[i].plot(sc.midpoints(sqw_c.coords["omega"]), sqw_c.data["q", i], color=colors[0])
    ax[i].errorbar(
        sc.midpoints(benzene.coords["omega"]).values,
        benzene.data["q", i].values,
        benzene.variances[i] ** 0.5,
        color=colors[1],
        alpha=0.4,
    )
    ax[i].set_title(f"q_mid: {q_bins.values[i]:.2f}")
    ax[i].set_xlim(-0.4, 0.4)

The simulation doesn't look too bad now, except at higher Q, especially at around Q=1.2-1.4, the simulation is a very poor approximation of the experimental spectra, what do we think could be causing this?


To investigate further we can compare the spectra by a couple of model free methods: 

* the 'reduced second moment' a spectral moment, that gives an idea of spectral width as a function of Q without needing to do any dodgy model fitting (see [this paper](https://pubs.rsc.org/cp/article/18/26/17202/517076))

$$
m_2(Q) = \frac{\int_{-E_{\max}}^{E_{\max}} E^2\, S(Q,E)\, \mathrm{d}E}{\int_{-E_{\max}}^{E_{\max}} S(Q,E)\, \mathrm{d}E}.
$$


* Elastic peak integral - integrate the spectra around the spectral peak, gives information about structuring



Have a go at implementing each method below, if you get stuck (or can't be bothered the functions are defined in functions.py)



In [ ]:
from functions import second_moment_analyser, limit_integral

In [ ]:
sim_sm = second_moment_analyser(
    sc.midpoints(sqw_c.coords["omega"]).values, sqw_c.data.values
)
exp_sm = second_moment_analyser(
    sc.midpoints(benzene.coords["omega"]).values,
    benzene.data.values,
    benzene.variances**0.5,
)


sim_int = limit_integral(
    sc.midpoints(sqw_c.coords["omega"]).values, sqw_c.data.values, limits=[-0.1, 0.1]
)
exp_int = limit_integral(
    sc.midpoints(benzene.coords["omega"]).values,
    benzene.data.values,
    limits=[-0.1, 0.1],
)


fig, ax = plt.subplots(1, 2, figsize=(6, 3))

ax[0].plot(sc.midpoints(q_bins), sim_sm, color=colors[0], label="Incoherent simulation")
ax[0].errorbar(
    sc.midpoints(q_bins).values,
    exp_sm[0],
    exp_sm[1],
    color=colors[1],
    label="experiment",
)


ax[1].plot(
    sc.midpoints(q_bins), sim_int, color=colors[0], label="Incoherent simulation"
)
ax[1].plot(sc.midpoints(q_bins).values, exp_int, color=colors[1], label="experiment")


ax[0].set_xlabel(r"Q / $Å^{-1}$")
ax[1].set_xlabel(r"Q / $Å^{-1}$")

ax[0].legend()

What do we think could be causing this localised Q dependent discrepency??? How could we test our hypothesis?


Have a crack if you fancy (or read the paper for spoilers)


Now let's load our fancy polarised QENS data taken from LET at ISIS, this means we can separate our incoherent and coherent data and look only at the incoherent!

Polarisation is great, except you lose loads of flux so measurements take a long time (although this is going to be much better at ESS with its mega pulse)

In [ ]:
LET_FOLDER = DATA / "LET/"
sys.path.append(str(LET_FOLDER))
from plet_data import load_plet_data

plet_inc = load_plet_data(f"{LET_FOLDER}/benzene_290_360_inc.nxspe", 3.60)
res = load_plet_data(f"{LET_FOLDER}/vanadium_362.nxspe", 3.62)

sqw_new = create_scipp_array(inc_dict["S"], inc_dict["q"], inc_dict["omega"])

q_bins_let = sc.linspace("q", 0.6, 1.8, num=20, unit="1/Angstrom")
omega_bins_let = sc.linspace("omega", -2.25, 2.25, num=300, unit="meV")
bin_dict_let = {"q": q_bins_let, "omega": omega_bins_let}


inc = sqw_new.rebin(**bin_dict_let)
plet_inc = plet_inc.rebin(**bin_dict_let)
res = res.rebin(**bin_dict_let)

inc_c = conv_and_norm(inc, res, plet_inc, bin_dict_let)

fig, ax = plt.subplots(5, 4, figsize=(10, 7.5))
ax = ax.ravel()

for i in range(len(q_bins_let) - 1):
    ax[i].plot(sc.midpoints(inc_c.coords["omega"]), inc_c.data["q", i], color=colors[0])
    ax[i].errorbar(
        sc.midpoints(plet_inc.coords["omega"]).values,
        plet_inc.data["q", i].values,
        plet_inc.variances[i] ** 0.5,
        color=colors[1],
        alpha=0.4,
    )
    ax[i].set_title(f"q_mid: {q_bins_let.values[i]:.2f}")
    ax[i].set_xlim(-2.15, 2.15)

ax[0].legend()

The agreement is dramatically improved! 


Perhaps try fitting the reduced second moment and elastic integral to this data, how does it compare?


Hooray for polarisation! now we can move on to trying to extract useful dynamical information from S(Q, $\omega$)